In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torch.nn.functional as F
from PIL import Image

import lpips

from diffusers.models import AutoencoderKL

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score, RocCurveDisplay
import seaborn as sns

from tqdm import tqdm

In [ ]:
class SafeSubset(Dataset):
    """
    Envuelve cualquier Subset (o Dataset) y garantiza que cada tensor
    devuelto es contiguo en memoria propia, evitando el error
    'Trying to resize storage that is not resizable'.
    """
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return img.clone().contiguous(), label


In [ ]:
test_dir = '/workspace/dataset/test/'
train_dir = '/workspace/dataset/train/'

IMG_SIZE   = 256
BATCH_SIZE = 16
VAL_SPLIT  = 0.2
SEED       = 42
N_SAMPLES  = 1024   # tamaño de los subsets pequeños para pruebas rápidas

resize_only = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

resize_augment = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=36),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomAutocontrast(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
train_ds_augmented = datasets.ImageFolder(root=train_dir, transform=resize_augment)
train_ds_clean     = datasets.ImageFolder(root=train_dir, transform=resize_only)
test_ds            = datasets.ImageFolder(root=test_dir,  transform=resize_only)

print(f"Clases detectadas : {train_ds_augmented.classes}")
print(f"class_to_idx      : {train_ds_augmented.class_to_idx}")

n_total = len(train_ds_augmented)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices = random_split(
    range(n_total), [n_train, n_val], generator=generator
)

real_label = train_ds_augmented.class_to_idx['real']

real_train_indices = [
    i for i in train_indices.indices
    if train_ds_augmented.samples[i][1] == real_label
]
real_val_indices = [
    i for i in val_indices.indices
    if train_ds_clean.samples[i][1] == real_label
]

train_dataset = Subset(train_ds_augmented, train_indices.indices)
val_dataset   = Subset(train_ds_clean,     val_indices.indices)

train_real_dataset = Subset(train_ds_augmented, real_train_indices)
val_real_dataset   = Subset(train_ds_clean,     real_val_indices)

print(f"Tamaños — train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_ds)}")
print(f"Tamaños reales — train_real: {len(train_real_dataset)}, val_real: {len(val_real_dataset)}")


In [ ]:
train_real_dataset = SafeSubset(train_real_dataset)
val_real_dataset = SafeSubset(val_real_dataset)

etiquetas = test_ds.classes
etiqueta_a_categoria = test_ds.class_to_idx

In [ ]:
img, lbl = train_real_dataset[15]
print(f"Shape imagen    : {img.shape}")
print(f"Rango píxeles   : {img.min():.2f} a {img.max():.2f}")
print(f"Categoría       : {lbl} ({train_ds_augmented.classes[lbl]})")
print(f"Clases → índice : {test_ds.class_to_idx}")


In [ ]:
dataloader_test = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

dataloader_train_real = DataLoader(
    train_real_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

dataloader_val_real = DataLoader(
    val_real_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Batches — test: {len(dataloader_test)}, test_small: {len(dataloader_test)}")
print(f"Batches — train_real: {len(dataloader_train_real)}, train_real_small: {len(dataloader_train_real)}")
print(f"Batches — val_real: {len(dataloader_val_real)}, val_real_small: {len(dataloader_val_real)}")

In [ ]:
from diffusers.models import AutoencoderKL
from torchsummary import summary

vae = AutoencoderKL.from_pretrained("REPA-E/e2e-invae-hf")
vae.eval().to("cuda")

In [ ]:
def normalizar(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def error_latente_robusto(mu, mu_real_media, mu_real_std, device):
    """
    Mide cuánto se aleja mu de la distribución aprendida de imágenes reales.
    Equivale a una distancia de Mahalanobis simplificada asumiendo
    independencia entre dimensiones.
    """
    mu_flat = mu.view(mu.size(0), -1)  # [B, C*H*W]

    mu_media = mu_real_media.to(device)
    mu_std   = mu_real_std.to(device).clamp(min=1e-6)

    distancia = ((mu_flat - mu_media) ** 2 / mu_std ** 2).mean(dim=1)
    return distancia

In [ ]:
vae.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

y_true = []


mus_reales = []

with torch.no_grad():
    for imgs, _ in tqdm(dataloader_train_real):
        imgs = imgs.to(device)
        mu = vae.encode(imgs).latent_dist.mean
        mus_reales.append(mu.view(mu.size(0), -1).cpu())

mus_reales = torch.cat(mus_reales, dim=0)

mu_real_media = mus_reales.mean(dim=0)
mu_real_std   = mus_reales.std(dim=0)

print(f"Distribución real en latente: media={mu_real_media.mean():.4f}, std={mu_real_std.mean():.4f}")


In [ ]:
vae.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

y_true = []
errores_latent_rob_list = []
errores_pixel = []

with torch.no_grad():
    for img, label in tqdm(dataloader_test):
        img = img.to(device)

        posterior = vae.encode(img).latent_dist
        mu        = posterior.mean
        
        error_latente_rob = error_latente_robusto(mu, mu_real_media, mu_real_std, device)
        pred = vae.decode(mu).sample

        errores_latent_rob_list.extend(error_latente_rob.cpu().tolist())
        loss_per_image = F.mse_loss(pred, img, reduction='none').mean(dim=[1,2,3])
        errores_pixel.extend(loss_per_image.cpu().tolist())

        y_true.extend(label.tolist())

In [ ]:
errores_latent_rob_list = np.array(errores_latent_rob_list)
errores_pixel = np.array(errores_pixel)
y_true = np.array(y_true)

plt.figure(figsize=(15, 5))

plt.hist(errores_latent_rob_list[y_true == 0], bins=300, alpha=0.5, label="fake")
plt.hist(errores_latent_rob_list[y_true == 1], bins=300, alpha=0.5, label="real")
plt.legend()
plt.title("Distribución de los errores de VAE preentrenado")
plt.xlabel("Error de reconstrucción en el espacio latente")
plt.ylabel("Conteo");
plt.savefig('/workspace/imagenes_metricas_vae/distribucion_errores_latent_vae_pretrained_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report


fake = 0
real = 1

# Rango de umbrales a considerar
umbral_min = np.min(errores_latent_rob_list[y_true==fake])
umbral_max = np.mean(errores_latent_rob_list[y_true==fake]) + 5*np.std(errores_latent_rob_list[y_true==fake])
print(umbral_min, umbral_max)
umbrales = np.linspace(umbral_min, umbral_max, 100)

# Iterar por cada umbral, calcular puntaje F1 y almacenar
scores = []
for umbral in umbrales:
    y_pred = [real if error < umbral else fake for error in errores_latent_rob_list]
    y_pred = np.array(y_pred)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    macro_f1 = report['macro avg']['f1-score']
    scores.append(macro_f1)

# Escoger el mejor umbral, el que arroje un F1 máximo
mejor_idx = np.argmax(scores)
mejor_umbral = umbrales[mejor_idx]
mejor_f1 = scores[mejor_idx]

print(f"Mejor umbral: {mejor_umbral:.4f}")
print(f"Mejor puntaje F1: {mejor_f1:.4f}")

In [ ]:
fake = 0
real = 1
y_pred = [real if error < mejor_umbral else fake for error in errores_latent_rob_list]
y_pred = np.array(y_pred)

In [ ]:
print("Distribución y_true:", np.bincount(y_true))
print("Distribución y_pred:", np.bincount(y_pred))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=etiquetas,
    yticklabels=etiquetas
)
plt.xlabel('Predicho', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title('Matriz de Confusión — VAE preentrenado - error espacio latente', fontsize=13)
plt.tight_layout()
plt.savefig('/workspace/imagenes_metricas_vae/confusion_matrix_latent_vae_pretrained_final.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=etiquetas))

In [ ]:
report_dict = classification_report(y_true, y_pred, target_names=etiquetas, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=df_report.values,
                 colLabels=df_report.columns,
                 rowLabels=df_report.index,
                 cellLoc='center',
                 loc='center')

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)

plt.savefig('/workspace/imagenes_metricas_vae/classification_report_latent_vae_pretrained_final.png', bbox_inches='tight', dpi=300)
print("Imagen guardada como classification_report.png")

In [ ]:
fpr, tpr, _ = roc_curve(y_true, [-e for e in errores_latent_rob_list])
auc_score   = roc_auc_score(y_true, [-e for e in errores_latent_rob_list])

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'VAE (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC — VAE preentrenado - error espacio latente', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/workspace/imagenes_metricas_vae/roc_curve_latent_vae_pretrained_final.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC final (test completo): {auc_score:.4f}")

In [ ]:
plt.figure(figsize=(15, 5))

plt.hist(errores_pixel[y_true == 0], bins=300, alpha=0.5, label="fake")
plt.hist(errores_pixel[y_true == 1], bins=300, alpha=0.5, label="real")
plt.legend()
plt.title("Distribución de los errores de VAE preentrenado")
plt.xlabel("Error de reconstrucción MSE")
plt.ylabel("Conteo");
plt.savefig('/workspace/imagenes_metricas_vae/distribucion_errores_MSE_vae_pretrained_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

fake = 0
real = 1

umbral_min = np.min(errores_pixel[y_true==fake])
umbral_max = np.mean(errores_pixel[y_true==fake]) + 5*np.std(errores_pixel[y_true==fake])
print(umbral_min, umbral_max)
umbrales = np.linspace(umbral_min, umbral_max, 100)

scores = []
for umbral in umbrales:
    y_pred = [real if error < umbral else fake for error in errores_pixel]
    y_pred = np.array(y_pred)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    macro_f1 = report['macro avg']['f1-score']
    scores.append(macro_f1)

mejor_idx = np.argmax(scores)
mejor_umbral = umbrales[mejor_idx]
mejor_f1 = scores[mejor_idx]

print(f"Mejor umbral: {mejor_umbral:.4f}")
print(f"Mejor puntaje F1: {mejor_f1:.4f}")

In [ ]:
fake = 0
real = 1
y_pred = [real if error < mejor_umbral else fake for error in errores_pixel]
y_pred = np.array(y_pred)

In [ ]:
print("Distribución y_true:", np.bincount(y_true))
print("Distribución y_pred:", np.bincount(y_pred))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=etiquetas,
    yticklabels=etiquetas
)
plt.xlabel('Predicho', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title('Matriz de Confusión — VAE preentrenado - error MSE', fontsize=13)
plt.tight_layout()
plt.savefig('/workspace/imagenes_metricas_vae/confusion_matrix_mse_vae_pretrained_final.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=etiquetas))

In [ ]:
report_dict = classification_report(y_true, y_pred, target_names=etiquetas, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=df_report.values,
                 colLabels=df_report.columns,
                 rowLabels=df_report.index,
                 cellLoc='center',
                 loc='center')

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)

plt.savefig('/workspace/imagenes_metricas_vae/classification_report_mse_vae_pretrained_final.png', bbox_inches='tight', dpi=300)
print("Imagen guardada como classification_report.png")

In [ ]:
fpr, tpr, _ = roc_curve(y_true, [-e for e in errores_pixel])
auc_score   = roc_auc_score(y_true, [-e for e in errores_pixel])

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'VAE (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC — VAE preentrenado - error MSE', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/workspace/imagenes_metricas_vae/roc_curve_mse_vae_pretrained_final.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC final (test completo): {auc_score:.4f}")